In [1]:
import os
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/home/junsu/tf-env/lib/python3.12/site-packages/nvidia/cuda_nvcc'
os.environ['TF_USE_LEGACY_KERAS'] = '1'

2026-03-24 00:09:08.782477: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-24 00:09:08.822942: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-24 00:09:09.585881: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2026-03-24 00:09:10.330087: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-24 00:09:10.362697: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-24 00:09:10.362755: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [2]:
!pip install transformers
!pip install gdown

In [3]:
!pip install pandas
import pandas as pd

In [4]:
# 학습   데이터   다운로드
!gdown https://drive.google.com/uc?id=13l621lx2nSnXpFpzh78UUEyds_DAzyn6
# 테스트   데이터   다운로드
!gdown https://drive.google.com/uc?id=10LwhiPlgjOZbtF0Bv5395wYIm23y_QfT

Downloading...
From (original): https://drive.google.com/uc?id=13l621lx2nSnXpFpzh78UUEyds_DAzyn6
From (redirected): https://drive.google.com/uc?id=13l621lx2nSnXpFpzh78UUEyds_DAzyn6&confirm=t&uuid=327f3ffc-c70a-4493-83c9-42fca40459cf
To: /mnt/c/Users/okss2/summ_train.json
100%|██████████████████████████████████████| 1.16G/1.16G [00:21<00:00, 54.8MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=10LwhiPlgjOZbtF0Bv5395wYIm23y_QfT
From (redirected): https://drive.google.com/uc?id=10LwhiPlgjOZbtF0Bv5395wYIm23y_QfT&confirm=t&uuid=bffd51ff-0f31-45a5-a533-a1f954924622
To: /mnt/c/Users/okss2/summ_test.json
100%|████████████████████████████████████████| 147M/147M [00:02<00:00, 51.7MB/s]


In [5]:
DATA_TRAIN_PATH = 'summ_train.json'
train_df = pd.read_json(DATA_TRAIN_PATH)
train_df = train_df.dropna()
train_df = train_df[:40000]
print('학 습   데 이 터 의   개 수 :', len(train_df))

DATA_TEST_PATH = 'summ_test.json'
test_df = pd.read_json(DATA_TEST_PATH)
test_df = test_df.dropna()
test_df = test_df[:5000]
print('테 스 트 데 이 터 의 개 수 :', len(test_df))

학 습   데 이 터 의   개 수 : 40000
테 스 트 데 이 터 의 개 수 : 5000


In [6]:
train_df.head()

,name,delivery_date,documents
0,문서요약 프로젝트,2020-12-23 12:01:15,"{'id': '290741778', 'category': '종합', 'media_t..."
1,문서요약 프로젝트,2020-12-23 12:01:15,"{'id': '290741792', 'category': '종합', 'media_t..."
2,문서요약 프로젝트,2020-12-23 12:01:15,"{'id': '290741793', 'category': '스포츠', 'media_..."
3,문서요약 프로젝트,2020-12-23 12:01:15,"{'id': '290741794', 'category': '정치', 'media_t..."
4,문서요약 프로젝트,2020-12-23 12:01:15,"{'id': '290741797', 'category': '종합', 'media_t..."


In [7]:
test_df.head()

,name,delivery_date,documents
0,문서요약 프로젝트,2020-12-23 12:01:15,"{'id': '340626877', 'category': '정치', 'media_t..."
1,문서요약 프로젝트,2020-12-23 12:01:15,"{'id': '340626896', 'category': '종합', 'media_t..."
2,문서요약 프로젝트,2020-12-23 12:01:15,"{'id': '340626904', 'category': 'IT,과학', 'medi..."
3,문서요약 프로젝트,2020-12-23 12:01:15,"{'id': '340627450', 'category': '사회', 'media_t..."
4,문서요약 프로젝트,2020-12-23 12:01:15,"{'id': '340627465', 'category': '경제', 'media_t..."


In [8]:
def preprocess_data(data):
    outs = []
    for doc in data['documents']:
        line = []
        line.append(doc['media_name'])
        line.append(doc['id'])
        para = []
        for sent in doc['text']:
            for s in sent:
                para.append(s['sentence'])
        line.append(para)
        line.append(doc['abstractive'][0])
        line.append(doc['extractive'])
        a = doc['extractive']
        if a[0] == None or a[1] == None or a[2] == None:
            continue
        outs.append(line)
    outs_df = pd.DataFrame(outs)
    outs_df.columns = ['media', 'id', 'article_original', 'abstractive', 'extractive']
    return outs_df

In [9]:
# 원문과 요약문을 각각 'article_original'와 'abstractive'열에 저장
train_data = preprocess_data(train_df)
train_data.head()

,media,id,article_original,abstractive,extractive
0,광양신문,290741778,"[ha당 조사료 400만원…작물별 차등 지원, 이성훈 sinawi@hanmail.n...",전라남도가 쌀 과잉문제를 근본적으로 해결하기 위해 올해부터 벼를 심었던 논에 벼 대...,"[2, 3, 10]"
1,광양신문,290741792,"[8억 투입, 고소천사벽화·자산마을에 색채 입혀, 이성훈 sinawi@hanmail...",여수시는 컬러빌리지 사업에 8억원을 투입하여 ‘색채와 빛’ 도시를 완성하여 고소천사...,"[2, 4, 11]"
2,광양신문,290741793,"[전남드래곤즈 해맞이 다짐…선수 영입 활발, 이성훈 sinawi@hanmail.ne...",전남드래곤즈 임직원과 선수단이 4일 구봉산 정상에 올라 일출을 보며 2018년 구단...,"[3, 5, 7]"
3,광양신문,290741794,"[11~24일, 매실·감·참다래 등 지역특화작목, 이성훈 sinawi@hanmail...","광양시는 농업인들의 경쟁력을 높이고, 소득안정을 위해 매실·감·참다래 등 지역특화작...","[2, 3, 4]"
4,광양신문,290741797,"[홍콩 크루즈선사‘아쿠아리우스’ 4, 6월 여수항 입항, 이성훈 sinawi@han...",올해 4월과 6월 두 차례에 걸쳐 타이완의 크루즈 관광객 4000여명이 여수에 입항...,"[3, 7, 4]"


In [10]:
# train_data의 첫 번째 샘플의 article_original 열의 값 출력
train_data['article_original'].loc[0]

['ha당 조사료 400만원…작물별 차등 지원',
 '이성훈 sinawi@hanmail.net',
 '전라남도가 쌀 과잉문제를 근본적으로 해결하기 위해 올해부터 시행하는 쌀 생산조정제를 적극 추진키로 했다.',
 '쌀 생산조정제는 벼를 심었던 논에 벼 대신 사료작물이나 콩 등 다른 작물을 심으면 벼와의 일정 소득차를 보전해주는 제도다.',
 '올해 전남의 논 다른 작물 재배 계획면적은 전국 5만ha의 약 21%인 1만 698ha로, 세부시행지침을 확정, 시군에 통보했다.',
 '지원사업 대상은 2017년산 쌀 변동직불금을 받은 농지에 10a(300평) 이상 벼 이외 다른 작물을 재배한 농업인이다.',
 '지원 대상 작물은 1년생을 포함한 다년생의 모든 작물이 해당되나 재배 면적 확대 시 수급과잉이 우려되는 고추, 무, 배추, 인삼, 대파 등 수급 불안 품목은 제외된다.',
 '농지의 경우도 이미 다른 작물 재배 의무가 부여된 간척지, 정부매입비축농지, 농진청 시범사업, 경관보전 직불금 수령 농지 등은 제외될 예정이다.',
 'ha(3000평)당 지원 단가는 평균 340만원으로 사료작물 400만원, 일반작물은 340만원, 콩·팥 등 두류작물은 280만원 등이다.',
 '벼와 소득차와 영농 편이성을 감안해 작물별로 차등 지원된다.',
 '논에 다른 작물 재배를 바라는 농가는 오는 22일부터 2월 28일까지 농지 소재지 읍면동사무소에 신청해야 한다.',
 '전남도는 도와 시군에 관련 기관과 농가 등이 참여하는‘논 타작물 지원사업 추진협의회’를 구성, 지역 특성에 맞는 작목 선정 및 사업 심의 등을 본격 추진할 방침이다.',
 '최향철 전라남도 친환경농업과장은 “최근 쌀값이 다소 상승추세에 있으나 매년 공급과잉에 따른 가격 하락으로 쌀농가에 어려움이 있었다”며“쌀 공급과잉을 구조적으로 해결하도록 논 타작물 재배 지원사업에 많이 참여해주길 바란다”고 말했다.']

In [11]:
test_data = preprocess_data(test_df)

In [12]:
train_data['news'] = train_data['article_original'].apply(lambda x: ' '.join(x))
test_data['news'] = test_data['article_original'].apply(lambda x: ' '.join(x))

In [13]:
train_data[['news', 'abstractive']].head()

,news,abstractive
0,ha당 조사료 400만원…작물별 차등 지원 이성훈 sinawi@hanmail.net...,전라남도가 쌀 과잉문제를 근본적으로 해결하기 위해 올해부터 벼를 심었던 논에 벼 대...
1,"8억 투입, 고소천사벽화·자산마을에 색채 입혀 이성훈 sinawi@hanmail.n...",여수시는 컬러빌리지 사업에 8억원을 투입하여 ‘색채와 빛’ 도시를 완성하여 고소천사...
2,전남드래곤즈 해맞이 다짐…선수 영입 활발 이성훈 sinawi@hanmail.net ...,전남드래곤즈 임직원과 선수단이 4일 구봉산 정상에 올라 일출을 보며 2018년 구단...
3,"11~24일, 매실·감·참다래 등 지역특화작목 이성훈 sinawi@hanmail.n...","광양시는 농업인들의 경쟁력을 높이고, 소득안정을 위해 매실·감·참다래 등 지역특화작..."
4,"홍콩 크루즈선사‘아쿠아리우스’ 4, 6월 여수항 입항 이성훈 sinawi@hanma...",올해 4월과 6월 두 차례에 걸쳐 타이완의 크루즈 관광객 4000여명이 여수에 입항...


In [14]:
# pytorch 설치
# !pip install torch

In [15]:
pip show transformers

Name: transformers
Version: 4.37.0
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /home/junsu/tf-env/lib/python3.12/site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [16]:
# 다운그레이드 진행
# !pip install transformers==4.40.0

In [17]:
# 버전맞추기..
!pip install transformers==4.37.0

In [18]:
# torch transfomer 충돌로 지움
# pip uninstall torch -y

In [19]:
# 정수 인코딩을 위한 DataSet 생성 (TFBartForConditionalGeneration > BartForConditionalGeneration)26.03
import tensorflow as tf
# from transformers import BartForConditionalGeneration, BartTokenizerFast
from transformers import TFBartForConditionalGeneration, BartTokenizerFast
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecay
import numpy as np
from tqdm import tqdm


In [20]:
import numpy as np
import tensorflow as tf

class KoBARTSummaryDataset(tf.keras.utils.Sequence):
    def __init__(self, df, tokenizer, max_len, batch_size, ignore_index=-100):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.docs = df
        self.batch_size = batch_size
        self.ignore_index = ignore_index  # 무시할 레이블의 인덱스 값. 기본값은 -100.
        self.indices = list(range(len(self.docs)))

    def __len__(self):
        return len(self.indices) // self.batch_size

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("Index out of range")
        batch_indices = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch = self.docs.iloc[batch_indices]
        input_ids = []
        decoder_input_ids = []
        labels = []
        for _, instance in batch.iterrows():
            # 'news' 열의 텍스트를 정수 인코딩하여 입력에 해당하는 'input_ids' 생성
            encoded_input = self.tokenizer.encode(instance['news'], max_length=self.max_len, padding='max_length', truncation=True)
            input_ids.append(encoded_input)
            # 'abstractive' 열의 텍스트를 정수 인코딩하여 레이블 생성
            encoded_label = self.tokenizer.encode(instance['abstractive'], max_length=self.max_len, padding='max_length', truncation=True)
            decoder_input = [self.tokenizer.eos_token_id] + encoded_label[:-1]
            decoder_input_ids.append(decoder_input)
            # 레이블에 -100을 이용하여 패딩을 적용
            label = encoded_label + [self.ignore_index] * (self.max_len - len(encoded_label))
            labels.append(label)
        return {
            'input_ids': np.array(input_ids),
            'decoder_input_ids': np.array(decoder_input_ids),
            'labels': np.array(labels)
        }

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

In [21]:
# 모델선언 (TF 빼기)
model = TFBartForConditionalGeneration.from_pretrained('gogamza/kobart-base-v1', from_pt=True)
tokenizer = BartTokenizerFast.from_pretrained('gogamza/kobart-base-v1')

# model = BartForConditionalGeneration.from_pretrained('gogamza/kobart-base-v1')
# tokenizer = BartTokenizerFast.from_pretrained('gogamza/kobart-base-v1')

/home/junsu/tf-env/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


AttributeError: 'NoneType' object has no attribute 'items'

In [ ]:
# 하이퍼파라미터 설정
batch_size = 2
max_len = 512
lr = 3e-5
max_epochs = 2
warmup_ratio = 0.1

In [ ]:
# Prepare datasets
train_dataset = KoBARTSummaryDataset(train_data, tokenizer, max_len=max_len, batch_size=batch_size)
test_dataset = KoBARTSummaryDataset(test_data, tokenizer, max_len=max_len, batch_size=batch_size)

total_steps = len(train_dataset) * max_epochs
warmup_steps = int(total_steps * warmup_ratio)

lr_schedule = CosineDecay(initial_learning_rate=lr, decay_steps=total_steps)
optimizer = Adam(learning_rate=lr_schedule)

In [ ]:
print('첫 번째 샘플의 원문 텍스트 :', train_data['news'].loc[0])

In [ ]:
print('첫 번째 샘플의 원문 텍스트의 정수 인코딩 및 패딩 결과 :', train_dataset[0]['input_ids'][0])
print('정수 인코딩 및 패딩 후의 길이 :', len(train_dataset[0]['input_ids'][0]))

In [ ]:
print('첫번째 샘플의 요약 문 :', train_data['abstractive'].loc[0])

In [ ]:
print('첫 번째 샘플의 요약문 텍스트의 정수 인코딩 및 패딩 결과 :', train_dataset[0]['decoder_input_ids'][0])
print('정수 인코딩 및 패딩 후의 길이 :', len(train_dataset[0]['decoder_input_ids'][0]))

In [ ]:
# -100의 값은 tokenizer.decode하면 에러나므로 임시로 0으로 변경 후 출력
test_array = train_dataset[0]['labels'][0]
test_array[test_array == -100] = 0
print('첫 번째 샘플의 요약문 레이블 :', tokenizer.decode(test_array))

In [ ]:
print(type(model))

In [ ]:
# 학습
# best_loss는 초기값으로 float('inf'), 즉 무한대로 초기화.
best_loss = float('inf')

for epoch in range(max_epochs):
    print(f'에포크 {epoch+1}/{max_epochs}')

    ### 학습 단계
    # 에포크 동안 누적된 총 손실을 저장할 변수
    total_train_loss = 0.0

    # 데이터를 배치 크기만큼 꺼내서 각 배치에 대해 모델을 학습
    for batch in tqdm(train_dataset, total=len(train_dataset), desc="훈련 중"):
        with tf.GradientTape() as tape:
            outputs = model(batch, training=True)
            loss = outputs.loss
        # 모델의 손실(outputs.loss) 값으로부터 역전파를 수행하여 파라미터를 업데이트.
        total_train_loss += tf.reduce_mean(loss).numpy()
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    # 한 에포크 동안 모든 배치에서 발생한 손실을 배치 수로 나누어,
    # 한 에포크 동안 배치 크기만큼의 데이터를 넣었을 때 발생한 평균 손실을 계산
    avg_train_loss = total_train_loss / len(train_dataset)
    print(f'훈련 손실: {avg_train_loss:.4f}')

    ### 평가 단계
    total_val_loss = 0.0
    for batch in tqdm(test_dataset, total=len(test_dataset), desc="검증 중"):
        outputs = model(batch, training=False)
        total_val_loss += tf.reduce_mean(outputs.loss).numpy()

    avg_val_loss = total_val_loss / len(test_dataset)
    print(f'검증 손실: {avg_val_loss:.4f}')

    # 최고 성능 모델 저장
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        model.save_pretrained('model')
        print(f'검증 손실이 {best_loss:.4f}로 개선되었습니다. 체크포인트를 저장했습니다.')

    # 에포크 종료 시 데이터 셔플
    train_dataset.on_epoch_end()
    test_dataset.on_epoch_end()

print("훈련이 완료되었습니다.")